# 🎙️ Nexlio Voice Studio — AI Voiceovers & Voice Cloning

Free text-to-speech, voice design and voice cloning on Google Colab's free T4 GPU.

**Quick start**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Open the **gradio.live** link printed by the last cell

⚠️ Only clone your own voice or voices you have clear permission to use.


In [ ]:
#@title 1️⃣ Install packages
!pip install -q -U voxcpm gradio soundfile
print("✅ Install done. If Colab asks to restart the session, restart it and run from step 2.")
# Note: a red 'pip dependency resolver' message about gcsfs/fsspec is only a warning — ignore it.


In [ ]:
#@title 2️⃣ Save model to Google Drive (optional, faster next time)
use_drive = True #@param {type:"boolean"}
import os
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/NexlioVoiceStudio_cache'
else:
    CACHE_DIR = '/content/hf_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE_DIR
print("✅ Model cache:", CACHE_DIR)


In [ ]:
#@title 3️⃣ Check GPU
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → T4 GPU, then run again.")


In [ ]:
#@title 4️⃣ Load VoxCPM2 (first time downloads a few GB)
from voxcpm import VoxCPM
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SR = model.tts_model.sample_rate
print(f"✅ Model loaded. Sample rate: {SR} Hz")


In [ ]:
#@title 5️⃣ Launch the app
import gradio as gr, numpy as np, torch, traceback

def _run(text, style="", ref=None, prompt_wav=None, prompt_text=None, cfg=2.0, steps=10):
    text = (text or "").strip()
    if not text:
        raise gr.Error("Please type some text.")
    style = (style or "").strip()
    if style:
        text = f"({style}){text}"
    try:
        wav = model.generate(
            text=text,
            reference_wav_path=ref,
            prompt_wav_path=prompt_wav,
            prompt_text=prompt_text,
            cfg_value=float(cfg),
            inference_timesteps=int(steps),
        )
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"Generation failed: {e}")
    finally:
        torch.cuda.empty_cache()
    return (SR, np.asarray(wav, dtype=np.float32))

def tts(text, cfg, steps):
    return _run(text, cfg=cfg, steps=steps)

def design(desc, text, cfg, steps):
    if not (desc or "").strip():
        raise gr.Error("Describe the voice, e.g. 'deep calm male voice, slow'.")
    return _run(text, style=desc, cfg=cfg, steps=steps)

def clone(ref, text, style, cfg, steps):
    if not ref:
        raise gr.Error("Upload or record a reference voice (5–20 seconds, clean).")
    return _run(text, style=style, ref=ref, cfg=cfg, steps=steps)

def ultimate(ref, ref_text, text, cfg, steps):
    if not ref:
        raise gr.Error("Upload or record a reference voice.")
    if not (ref_text or "").strip():
        raise gr.Error("Type the exact words spoken in the reference audio.")
    return _run(text, ref=ref, prompt_wav=ref, prompt_text=ref_text.strip(), cfg=cfg, steps=steps)

def knobs():
    with gr.Accordion("Advanced", open=False):
        cfg = gr.Slider(1.0, 3.0, value=2.0, step=0.1, label="CFG (higher = follows voice/style more strictly)")
        steps = gr.Slider(4, 30, value=10, step=1, label="Steps (higher = better quality, slower)")
    return cfg, steps

with gr.Blocks(title="Nexlio Voice Studio") as app:
    gr.Markdown("# 🎙️ Nexlio Voice Studio\nAI voiceovers for ads · by [Nexlio](https://nexlio.cc) · 30 languages · 48 kHz. "
                "Only clone voices you own or have permission to use.")

    with gr.Tab("🗣️ Text to Speech"):
        t = gr.Textbox(label="Text", lines=4, value="Grow your business with Nexlio. Google, Facebook and TikTok ads that bring real customers.")
        c, s = knobs()
        b = gr.Button("Generate", variant="primary")
        o = gr.Audio(label="Output")
        b.click(tts, [t, c, s], o)

    with gr.Tab("🎨 Voice Design"):
        d = gr.Textbox(label="Voice description", value="A young woman, warm and gentle voice, slightly smiling")
        t = gr.Textbox(label="Text", lines=4, value="Looking for more customers in Dubai? Nexlio runs ads that work. Message us today!")
        c, s = knobs()
        b = gr.Button("Generate", variant="primary")
        o = gr.Audio(label="Output")
        b.click(design, [d, t, c, s], o)

    with gr.Tab("🧬 Voice Cloning"):
        r = gr.Audio(label="Reference voice (5–20 s, clean, one speaker)", type="filepath",
                     sources=["upload", "microphone"])
        t = gr.Textbox(label="Text to speak in that voice", lines=4)
        st = gr.Textbox(label="Style (optional)", placeholder="e.g. slightly faster, cheerful tone")
        c, s = knobs()
        b = gr.Button("Clone", variant="primary")
        o = gr.Audio(label="Output")
        b.click(clone, [r, t, st, c, s], o)

    with gr.Tab("🏆 Ultimate Cloning"):
        gr.Markdown("Best similarity: give the reference audio **and** its exact transcript.")
        r = gr.Audio(label="Reference voice", type="filepath", sources=["upload", "microphone"])
        rt = gr.Textbox(label="Exact transcript of the reference audio", lines=2)
        t = gr.Textbox(label="New text to speak", lines=4)
        c, s = knobs()
        b = gr.Button("Clone", variant="primary")
        o = gr.Audio(label="Output")
        b.click(ultimate, [r, rt, t, c, s], o)

app.queue().launch(share=True, debug=True)
